# JAX 源码整体架构

## 目标

建立 JAX 源码的**全局地图**——理解顶层模块组织、核心数据结构、关键入口函数。后续 notebook 再按模块深入。

## 阅读路线

```
整体结构 → 核心 IR → 编译链路 → Pallas (TPU kernel) → Mosaic (内存系统) → TPU runtime
```

## 1. JAX 包的物理结构

JAX 安装后的包目录布局（关键模块）：

```
jax/
├── _src/                        # 内部实现 (私有 API)
│   ├── core.py                  # Jaxpr IR、抽象值、基本原语
│   ├── linear_util.py           # 线性化 / JVP 工具
│   ├── dispatch.py              # 调度：Jaxpr → 后端执行
│   ├── api.py                   # jax.jit / jax.grad / jax.vmap 等公开 API 实现
│   ├── interpreters/            # 解释器 & 变换 (AD, batching, partial eval)
│   ├── pallas/                  # Pallas: TPU/GPU kernel 编程框架
│   │   ├── pallas_call.py       # pallas_call 入口 (49 KB)
│   │   ├── primitives.py        # Pallas 原语 (49 KB)
│   │   ├── core.py              # BlockSpec / GridSpec (63 KB)
│   │   └── mosaic/              # Mosaic: TPU 硬件后端
│   │       ├── lowering.py      # StableHLO → TPU (184 KB!)
│   │       ├── pipeline.py      # VMEM 双缓冲 (80 KB)
│   │       ├── primitives.py    # TPU 专用原语 (42 KB)
│   │       ├── tpu_info.py      # TPU 硬件信息 (22 KB)
│   │       └── core.py          # CompilerParams / MemorySpace (20 KB)
│   ├── lax/                     # LAX: JAX 的线性代数层
│   ├── numpy/                   # jax.numpy 实现
│   ├── sharding.py              # 分布式分片
│   └── tpu_custom_call.py       # TPU CustomCall 后端序列化
│
├── experimental/                # 实验性 API (薄封装，指向 _src)
│   ├── pallas/
│   │   ├── tpu.py               # pltpu.VMEM / CompilerParams
│   │   └── ops/                 # TPU 算子 (flash attention 等)
│   └── mosaic/
│
├── interpreters/                # 公开的解释器重导出
├── extend/                      # JAX 扩展 API (自定义后端/IFRT)
├── lax/                         # 公开 LAX API
├── numpy/                       # 公开 NumPy API
└── nn/                          # 公开 NN API
```

In [ ]:
import jax, os
jax_base = os.path.dirname(jax.__file__)

# 各子包的文件统计
key_dirs = ['_src', '_src/pallas', '_src/pallas/mosaic', '_src/interpreters', 'experimental', 'interpreters']
for d in key_dirs:
    path = os.path.join(jax_base, d)
    if os.path.isdir(path):
        py_files = [f for f in os.listdir(path) if f.endswith('.py') and not f.startswith('_')]
        total_kb = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path) if f.endswith('.py')) // 1024
        print(f'  {d:35s}  {len(py_files):3d} 个公开模块  {total_kb:5d} KB')

## 2. 架构分层

JAX 从用户代码到硬件执行，大约分 5 层：

```
┌──────────────────────────────────────────┐
│  User API: jit / grad / vmap / pallas    │  ← jax._src.api
├──────────────────────────────────────────┤
│  Tracing & Jaxpr IR                      │  ← jax._src.core / linear_util
│  将 Python 函数 trace 为纯函数 IR 图     │
├──────────────────────────────────────────┤
│  Transformations (AD / batching / eval)   │  ← jax._src.interpreters
│  对 Jaxpr 做自动微分、向量化、部分求值   │
├──────────────────────────────────────────┤
│  Lowering & Compilation                  │  ← jax._src.dispatch / pallas/mosaic
│  Jaxpr → StableHLO → HLO → 可执行文件    │
├──────────────────────────────────────────┤
│  Runtime: XLA / IFRT / PJRT              │  ← jax._src.xla_bridge / extend
│  设备管理、内存分配、kernel 执行         │
└──────────────────────────────────────────┘
```

## 3. 核心数据结构速览

理解 JAX 源码绕不开的 5 个核心类型：

In [ ]:
from jax._src.core import Jaxpr, ClosedJaxpr, Var, Literal
from jax._src.linear_util import WrappedFun
from jax._src.interpreters.partial_eval import Trace, trace_to_jaxpr_dynamic

# Jaxpr: JAX 的中间表示 (类似 TF Graph / Torch FX Graph)
# 结构字段说明:
print('Jaxpr 结构字段:')
print('  constvars  — 常量变量列表')
print('  invars     — 输入变量列表')
print('  outvars    — 输出变量列表')
print('  eqns       — 等式列表 (每个等式 = 一个原语调用)')
print('  effects    — 副作用集合 (如 debug print)')

print()
# 查看一个简单的 Jaxpr
def f(x):
    return x * 2 + 1

import jax
jaxpr = jax.make_jaxpr(f)(3.0)
print('jax.make_jaxpr(f)(3.0) 输出:')
print(jaxpr)

print()
# 等式的结构
if jaxpr.eqns:
    eq = jaxpr.eqns[0]
    print(f'第一条等式: primitive={eq.primitive}, invars={eq.invars}, outvars={eq.outvars}')

## 4. 关键入口函数

| 入口 | 位置 | 作用 |
|------|------|------|
| `jax.jit(f)` | `_src/api.py` → `_src/dispatch.py` | 编译并缓存函数 |
| `jax.grad(f)` | `_src/api.py` → `_src/interpreters/ad.py` | 自动微分变换 |
| `jax.vmap(f)` | `_src/api.py` → `_src/interpreters/batching.py` | 自动向量化 |
| `jax.make_jaxpr(f)` | `_src/api.py` → `_src/interpreters/partial_eval.py` | 生成 Jaxpr |
| `pl.pallas_call(...)` | `experimental/pallas/tpu.py` → `_src/pallas/pallas_call.py` | TPU kernel 入口 |
| `pltpu.CompilerParams(...)` | `experimental/pallas/tpu.py` → `_src/pallas/mosaic/core.py` | TPU 编译参数 |

调用链追踪示例：

In [ ]:
# 追踪 jax.jit 的调用链
import inspect

def trace_call_stack(func, depth=0, max_depth=3):
    """追踪一个函数的直接调用链"""
    if depth >= max_depth:
        return
    
    src_file = inspect.getsourcefile(func)
    src_line = inspect.getsourcelines(func)[1]
    print(f"{'  ' * depth}{func.__name__}() @ {src_file}:{src_line}")

# jax.jit 的实际入口
print('===== jax.jit 到 dispatch 的路径 =====')
from jax._src import api as jax_api
from jax._src import dispatch

print('jax.jit →')
print('  jax._src.api.jit()        # 用户 API 层')
print('  → _src/dispatch.py        # 调度与编译缓存')
print('  → _src/interpreters/xla.py  # XLA 后端执行')

## 5. 平台检测与分发

JAX 如何判断当前运行在什么设备上：

In [ ]:
# 平台检测链
print('=== 设备信息 ===')
for d in jax.devices():
    print(f'  {d.platform}  {d.device_kind}  id={d.id}')

print()

# TPU 平台检测路径
from jax._src.xla_bridge import get_backend
backend = get_backend('tpu')
print(f'Backend: {backend.platform}')
print(f'Device count: {backend.device_count()}')

## 6. 各模块的关系图

```
                 ┌─────────────────┐
                 │   User Code     │
                 └───────┬─────────┘
                         │
              ┌──────────▼──────────┐
              │  jax._src.api       │  jit / grad / vmap / pallas_call
              └──────────┬──────────┘
                         │
      ┌──────────────────┼──────────────────┐
      ▼                  ▼                  ▼
┌───────────┐  ┌──────────────────┐  ┌──────────────┐
│_src/core  │  │_src/interpreters │  │_src/dispatch │
│ Jaxpr IR  │  │ AD / vmap /      │  │ 编译调度      │
│ 抽象值    │  │ partial_eval     │  │               │
└─────┬─────┘  └────────┬─────────┘  └───────┬───────┘
      │                 │                    │
      └─────────────────┼────────────────────┘
                        │
              ┌─────────▼──────────┐
              │  XLA / PJRT        │
              │  (C++ runtime)     │
              └─────────┬──────────┘
                        │
              ┌─────────▼──────────┐
              │  TPU / GPU / CPU   │
              └────────────────────┘
```

## 下一步

→ `01-xla-compile-pipeline.ipynb` — 深入 JAX → StableHLO → HLO → TPU 二进制的完整编译链路